# 03. Multi-Agent Autonomous Simulations & Counterfactual Experiments

**Preset:** `SimulationPreset` + custom `SchemaPolicy` & `EpisodicScoringPolicy`

## Why `canonarc` Fits This Domain

Multi-agent social, economic, or survival simulations require:
1. **Per-Tick Structured World State:** Agents mutate energy, morale, resources, and directed affinity edges (`affinity`, `stance`) every tick.
2. **Partial Observability:** Agent A should not magically know about a secret sabotage witnessed only by Agent C—unless Agent C explicitly tells Agent A (`inform_entity(..., epistemic="told")`).
3. **Counterfactual "What-If" Forking:** Researchers want to fork a simulation at Tick $T$, inject a different intervention, and compare the resulting social graphs with `diff_timelines()`.
4. **Long-Horizon Memory Compaction:** `consolidate_episodes()` compacts noisy low-salience ticks into higher-level summaries so prompts stay lean over thousands of ticks.

In this notebook, we simulate a 3-agent polar outpost, customize `SchemaPolicy`, demonstrate per-agent POV filtering, run a counterfactual branch, and compact old episodes.

In [1]:
from canonarc import (
    EdgeDelta,
    EntityDelta,
    EntityRecord,
    EpisodeRecord,
    EpisodicScoringPolicy,
    InformDelta,
    CanonArcEngine,
    SchemaPolicy,
    SimulationPreset,
    StepDelta,
    VoiceExemplar,
)

# Customize SchemaPolicy & EpisodicScoringPolicy for our simulation
sim_policy = SchemaPolicy(
    hard_filter_keys=(),
    soft_boost_weights={"stance": 0.20, "urgency": 0.15},
    episodic_policy=EpisodicScoringPolicy(
        provenance_bonus=1.2,
        semantic_weight=0.5,
        base_decay_rate=0.02,
    ),
)

engine = CanonArcEngine(":memory:", preset=SimulationPreset(policy=sim_policy))
print("Simulation engine initialized with custom SchemaPolicy.")

Simulation engine initialized with custom SchemaPolicy.


## Step 1: Seed Outpost Agents & Initial Affinity Graph (Tick 0)

In [2]:
for aid, name, role in [
    ("agent_alice", "Alice", "medic"),
    ("agent_bob", "Bob", "engineer"),
    ("agent_cara", "Cara", "comms_officer"),
]:
    engine.register_entity(
        EntityRecord(
            entity_id=aid,
            canonical_name=name,
            base_attributes={"role": role},
            state={"energy": 85, "morale": "steady"},
        ),
        step=0,
    )

engine.commit_step(
    step=1,
    delta=StepDelta(
        edges=[
            EdgeDelta(source_id="agent_alice", target_id="agent_bob", state={"affinity": 0.70, "stance": "cooperative"}),
            EdgeDelta(source_id="agent_bob", target_id="agent_alice", state={"affinity": 0.70, "stance": "cooperative"}),
            EdgeDelta(source_id="agent_cara", target_id="agent_bob", state={"affinity": 0.55, "stance": "neutral"}),
        ]
    ),
)
print("Seeded 3 outpost agents and initial affinity edges.")

Seeded 3 outpost agents and initial affinity edges.


## Step 2: Simulate Ticks 5–25 (Routine Ticks, Aid, Secret Sabotage & Gossip)

- **Ticks 5 & 8:** Low-salience routine maintenance ticks (later candidates for `consolidate_episodes()`).
- **Tick 15:** Alice shares medical supplies with Bob (`witnesses=["agent_alice", "agent_bob"]`).
- **Tick 22:** Bob secretly siphons generator fuel; **only Cara** witnesses it (`witnesses=["agent_bob", "agent_cara"]`).
- **Tick 25:** Cara warns Alice secondhand via `InformDelta(epistemic="told")`.

In [3]:
# Routine low-salience ticks
engine.commit_step(
    step=5,
    delta=StepDelta(
        episodes=[
            EpisodeRecord(
                episode_id="ep_t05_inventory",
                predicate="routine",
                summary="Alice logged routine medical inventory in the storage bay.",
                participants={"agent_alice": "actor"},
                witnesses=["agent_alice"],
                salience=2.0,
            )
        ]
    ),
)
engine.commit_step(
    step=8,
    delta=StepDelta(
        episodes=[
            EpisodeRecord(
                episode_id="ep_t08_radio_check",
                predicate="routine",
                summary="Cara completed the daily weather beacon calibration.",
                participants={"agent_cara": "actor"},
                witnesses=["agent_cara"],
                salience=2.5,
            )
        ]
    ),
)

# Tick 15: Cooperative resource sharing
engine.commit_step(
    step=15,
    delta=StepDelta(
        entities=[EntityDelta(entity_id="agent_bob", state={"energy": 95, "morale": "grateful"})],
        edges=[
            EdgeDelta(
                source_id="agent_bob",
                target_id="agent_alice",
                state={"affinity": 0.90, "stance": "allied"},
                caused_by_episode_id="ep_t15_med_share",
                reason="Alice shared emergency medical supplies",
            )
        ],
        episodes=[
            EpisodeRecord(
                episode_id="ep_t15_med_share",
                predicate="item_transfer",
                summary="Alice shared her reserve medical supplies with Bob after the storm.",
                participants={"agent_alice": "giver", "agent_bob": "receiver"},
                witnesses=["agent_alice", "agent_bob"],
                salience=7.5,
            )
        ],
    ),
    exemplars=[
        VoiceExemplar(
            step=15,
            speaker_id="agent_alice",
            text="We should pool our rations and medical packs before the next whiteout hits the ridge.",
            metadata={"stance": "cooperative", "urgency": "high"},
            quality_score=9.0,
        )
    ],
)

# Tick 22: Secret fuel sabotage (witnessed ONLY by Cara and Bob, NOT Alice)
engine.commit_step(
    step=22,
    delta=StepDelta(
        edges=[
            EdgeDelta(
                source_id="agent_cara",
                target_id="agent_bob",
                state={"affinity": -0.60, "stance": "suspicious"},
                caused_by_episode_id="ep_t22_fuel_siphon",
                reason="Caught Bob siphoning reserve generator fuel",
            )
        ],
        episodes=[
            EpisodeRecord(
                episode_id="ep_t22_fuel_siphon",
                predicate="sabotage",
                summary="Bob secretly siphoned reserve generator fuel into a private canister.",
                participants={"agent_bob": "actor"},
                witnesses=["agent_bob", "agent_cara"],
                salience=9.2,
            )
        ],
    ),
)
print("Simulated ticks 5 through 22.")

Simulated ticks 5 through 22.


## Step 3: Verify Partial Observability (Tick 23 vs. Tick 26)

At **Tick 23**, Alice has **not** been told about the fuel sabotage, so querying `pov_entity_ids=["agent_alice"]` hides `ep_t22_fuel_siphon`. At **Tick 25**, Cara informs Alice secondhand (`epistemic="told"`), making it visible to Alice at **Tick 26**.

In [4]:
# Tick 23: Alice's POV does NOT know about the fuel sabotage
alice_t23 = engine.retrieve_context(
    step=23,
    mentions=["Alice", "Bob"],
    pov_entity_ids=["agent_alice"],
)
print("Alice POV episodes at Tick 23:", [ep.episode_id for ep in alice_t23.episodes])
assert "ep_t22_fuel_siphon" not in [ep.episode_id for ep in alice_t23.episodes]

# Tick 25: Cara informs Alice secondhand
engine.commit_step(
    step=25,
    delta=StepDelta(
        inform=[InformDelta(episode_id="ep_t22_fuel_siphon", entity_id="agent_alice", epistemic="told")],
        edges=[
            EdgeDelta(
                source_id="agent_alice",
                target_id="agent_bob",
                state={"affinity": -0.20, "stance": "guarded"},
                caused_by_episode_id="ep_t22_fuel_siphon",
                reason="Cara warned Alice about the siphoned fuel",
            )
        ],
    ),
)

# Tick 26: Now Alice sees the episode (marked 'told') and her updated guarded stance
alice_t26 = engine.retrieve_context(
    step=26,
    mentions=["Alice", "Bob"],
    pov_entity_ids=["agent_alice"],
)
print()
print(engine.format_context(alice_t26))
assert "ep_t22_fuel_siphon" in [ep.episode_id for ep in alice_t26.episodes]

Alice POV episodes at Tick 23: ['ep_t15_med_share', 'ep_t05_inventory']

=== Agent Simulation State (Tick 26) ===
Agent agent_alice (Alice): state={'energy': 85, 'morale': 'steady'}
Agent agent_bob (Bob): state={'energy': 95, 'morale': 'grateful'}
Relation [stance] agent_alice -> agent_bob: {'affinity': -0.2, 'stance': 'guarded'}
Relation [stance] agent_bob -> agent_alice: {'affinity': 0.9, 'stance': 'allied'}
Episodic Memory:
  [Tick 15 | item_transfer] Alice shared her reserve medical supplies with Bob after the storm. (known_by=['agent_alice', 'agent_bob'])
  [Tick 22 | sabotage] Bob secretly siphoned reserve generator fuel into a private canister. (known_by=['agent_bob', 'agent_cara', 'agent_alice'])
  [Tick 5 | routine] Alice logged routine medical inventory in the storage bay. (known_by=['agent_alice'])
Past Utterance Anchors:
  agent_alice: We should pool our rations and medical packs before the next whiteout hits the ridge.


## Step 4: Counterfactual Branching & Long-Horizon Episode Consolidation

1. **Counterfactual experiment:** Fork at Tick 20 (before the sabotage) into `"transparent_rationing"`, run an alternate Tick 22 where the team pools fuel openly, and diff `"main"` vs. `"transparent_rationing"` at Tick 30.
2. **Episode consolidation:** Compact old low-salience routine ticks (`salience <= 4.0`, `step < 20`) into a single summary episode so retrieval stays fast over long runs.

In [5]:
# 1. Counterfactual fork at Tick 20
engine.branch(at_step=20, name="transparent_rationing", switch=True)
engine.commit_step(
    step=22,
    delta=StepDelta(
        edges=[
            EdgeDelta(
                source_id="agent_cara",
                target_id="agent_bob",
                state={"affinity": 0.80, "stance": "cooperative"},
                reason="Bob proposed an open fuel ledger",
            )
        ],
        episodes=[
            EpisodeRecord(
                episode_id="ep_t22_open_ledger",
                predicate="agreement",
                summary="Bob and Cara instituted a transparent fuel ledger for the outpost.",
                participants={"agent_bob": "actor", "agent_cara": "partner"},
                witnesses=["agent_alice", "agent_bob", "agent_cara"],
                salience=8.0,
            )
        ],
    ),
)

diff = engine.diff_timelines("main", "transparent_rationing", at_step=30)
print("Divergent state keys between 'main' and 'transparent_rationing':")
for item in diff["changed"]:
    print(" -", item["subject_key"])

# 2. Compact low-salience routine ticks (< Tick 20) on 'main'
engine.checkout("main")
summary_ep = engine.consolidate_episodes(
    step=30,
    before_step=20,
    max_salience=4.0,
    summary="During Ticks 1–19, the outpost completed routine medical inventory and radio calibration.",
)
print("\nConsolidated episode:", summary_ep.episode_id, "compacted:", summary_ep.consolidated_from)
assert set(summary_ep.consolidated_from) == {"ep_t05_inventory", "ep_t08_radio_check"}

engine.close()

Divergent state keys between 'main' and 'transparent_rationing':
 - agent_alice:stance->agent_bob
 - agent_cara:stance->agent_bob

Consolidated episode: ep_consolidated_30_843055bf compacted: ['ep_t05_inventory', 'ep_t08_radio_check']
